# Metody wstępnego przetwarzania danych, semestr 2026L

## Lab 11. Metody wstępnego przetwarzania tekstu na potrzeby Natural Language Processing (NLP), część 2

W tej części zajmiemy się:
* dalszą eksploracją możliwości wyrażeń regularnych w kontekście wstępnego przetwarzania tekstu na potrzeby NLP,
* proste przykłady kolejnych etapów pracy z tekstem z wykorzystaniem bibliotek NLTK, spaCy

### 1. Oczyszczanie tekstu z wykorzystaniem wyrażeń regularnych

Poniżej rozszerzenie dostępnych metaznaków (lub ich inne formy) wyrażeń regularnych.

| Metaznak | Opis                                                                                     
|----------|------------------------------------------------------------------------------------------
| `\A`     | Zwraca dopasowanie, jeśli określone znaki znajdują się na początku ciągu
| `\b`     | Zwraca dopasowanie, jeśli określone znaki znajdują się na początku lub końcu wyrazu
| `\B`     | Zwraca dopasowanie, jeśli określone znaki występują, ale NIE na początku ani końcu wyrazu
| `\d`     | Zwraca dopasowanie, jeśli ciąg zawiera cyfry (liczby od 0 do 9), to samo co [0-9]
| `\D`     | Zwraca dopasowanie, jeśli ciąg NIE zawiera cyfr, to samo co [^0-9]
| `\s`     | Zwraca dopasowanie, jeśli ciąg zawiera znak białej spacji
| `\S`     | Zwraca dopasowanie, jeśli ciąg NIE zawiera znaku białej spacji
| `\w`     | Zwraca dopasowanie, jeśli ciąg zawiera dowolne znaki słowa (litery od a do Z, cyfry od 0 do 9 oraz znak podkreślenia `_`) (czyli [a-zA-Z0-9_])
| `\W`     | Zwraca dopasowanie, jeśli ciąg NIE zawiera żadnych znaków, które zwraca \w
| `\Z`     | Zwraca dopasowanie, jeśli określone znaki znajdują się na końcu ciągu

In [24]:
import re

**Przykład 1**

In [29]:
# pozbywanie się niechcianych znaków z tekstu
# można osiągnąć poprzez wskazanie tych znaków poprzez zbiór ([]) oraz dodanie oznaczenia dopełnienia (^) zbioru
# więc faktycznie szukamy wszystkich znaków, które nie są w zbiorze

text = "Nie wiem$    dlaczego Bob$22  niE przyszedł na ^spotkanie."
cleaned = re.findall(r'[^@#$!^\d]+', text)
print(cleaned)
print(''.join(cleaned))

['Nie wiem', '    dlaczego Bob', '  niE przyszedł na ', 'spotkanie.']
Nie wiem    dlaczego Bob  niE przyszedł na spotkanie.


**Przykład 2**

Funkcja **`re.sub(wyrażenie, zamień_na, tekst_do_przetworzenia)`** zamiania wszystkie wystąpienia odnalezionych dopasowań do wyrażenia regularnego na podany łańcuch znaków. Poniżej dokumentacja tej funkcji.

In [33]:
print(re.sub.__doc__)

Return the string obtained by replacing the leftmost
    non-overlapping occurrences of the pattern in string by the
    replacement repl.  repl can be either a string or a callable;
    if a string, backslash escapes in it are processed.  If it is
    a callable, it's passed the Match object and must return
    a replacement string to be used.


In [34]:
# nadmierna liczba białych znaków również nie jest czymś pożądanym
# zastępujemy 

clean_text = re.sub(r'\s{2,}', ' ', text).strip()
# albo
# clean_text = re.sub(r'\s+', ' ', text).strip()
print(clean_text)

Nie wiem$ dlaczego Bob$22 niE przyszedł na ^spotkanie.


**Przykład 3**

Uruchamianie wbudowanej metody `str.replace` dla każdego elementu może być dość czasochłonne w zapisie. Można sobie wyobrazić tablicę, w której przygotowujemy pary wartości: stara wartość -> zamień na i uruchomić pętlę. Nie musimy tego robić, gdyż istnieje wbudowane funkcje, które ten proces ułatwią.

Pierwsza z nich to `str.translate`, która oczekuje podania mapowania (np. słownika), który stanowi zbiór par opisanych powyżej. Minusem wykorzystania tylko tej funkcji jest konieczność określania znaków w postaci ich kodów z tablicy znaków.

In [37]:
print(str.translate.__doc__)

Replace each character in the string using the given translation table.

  table
    Translation table, which must be a mapping of Unicode ordinals to
    Unicode ordinals, strings, or None.

The table must implement lookup/indexing via __getitem__, for instance a
dictionary or list.  If this operation raises LookupError, the character is
left untouched.  Characters mapped to None are deleted.


In [39]:
ord('A'), ord('a')

(65, 97)

In [40]:
replace_table = {65: 97}

'Ala ma kota'.translate(replace_table)

'ala ma kota'

Dla ułatwienia całego procesu w API znajdziemy również wbudowaną funkcję `str.maketrans()`.

In [41]:
str.maketrans('A','a')

{65: 97}

In [42]:
# możemy więc wykorzystać ją w łańcuchu metod
'Ala ma kota'.translate(str.maketrans('A','a'))

'ala ma kota'

In [50]:
# ta funkcja tworzy translację dla każdego znaku w x na znak z y - obie sekwencje muszą być tej samej długości

x = "@#$%^&*"
y = "       "
mytable = str.maketrans(x, y)
print(text.translate(mytable))

Nie wiem     dlaczego Bob 22  niE przyszedł na  spotkanie.


In [53]:
# jeszcze jedna forma tej funkcji pozwala na wskazanie znaków, które mają być usunięte z tekstu
# co można wykorzystać podobnie do replace, ale dla wielu znaków jednocześnie
x = ''
y = ''
z = "@#$%^&*"
mytable = str.maketrans(x, y, z)
print(mytable)
print(text.translate(mytable))

{64: None, 35: None, 36: None, 37: None, 94: None, 38: None, 42: None}
Nie wiem    dlaczego Bob22  niE przyszedł na spotkanie.


In [54]:
# możemy to wykorzystać np. do pozbycia się wszystkich znaków interpunkcji
text = "Hej! Jak się masz? U mnie wszystko OK."
text.translate(str.maketrans('', '', string.punctuation))

'Hej Jak się masz U mnie wszystko OK'

**Przykład 4**

In [55]:
# wykorzystując wyrażenia regularne można bardzo prostym wzorcem
# wyeliminować wszystko to co nie jest słowem (patrz tabela) oraz białym znakiem

clean_text = re.sub(r'[^\w\s]', '', text)
print(clean_text)

Hej Jak się masz U mnie wszystko OK


**Przykład 5**

Teraz zajmiemy się emotikonami, które również mogą pojawić się w tekście, a ich pojawienie się może być przydatne, np. w zadaniach analizy sentymentu.
Wykorzystamy istniejącą bibliotekę `emoji`.

In [ ]:
!pip install emoji

In [58]:
import emoji

emoji_text = "Kawa zaparzona ☕, plany na dziś zrobione 📝. Czas ruszać do działania i łapać chwile! ✨ Życzę Wam wspaniałego i pełnego energii dnia! 💪😊"

emoji_corpus = [emoji.demojize(doc) for doc in emoji_text]
print("Tekst bez emoji:\n", emoji_corpus)

Tekst bez emoji:
 ['K', 'a', 'w', 'a', ' ', 'z', 'a', 'p', 'a', 'r', 'z', 'o', 'n', 'a', ' ', ':hot_beverage:', ',', ' ', 'p', 'l', 'a', 'n', 'y', ' ', 'n', 'a', ' ', 'd', 'z', 'i', 'ś', ' ', 'z', 'r', 'o', 'b', 'i', 'o', 'n', 'e', ' ', ':memo:', '.', ' ', 'C', 'z', 'a', 's', ' ', 'r', 'u', 's', 'z', 'a', 'ć', ' ', 'd', 'o', ' ', 'd', 'z', 'i', 'a', 'ł', 'a', 'n', 'i', 'a', ' ', 'i', ' ', 'ł', 'a', 'p', 'a', 'ć', ' ', 'c', 'h', 'w', 'i', 'l', 'e', '!', ' ', ':sparkles:', ' ', 'Ż', 'y', 'c', 'z', 'ę', ' ', 'W', 'a', 'm', ' ', 'w', 's', 'p', 'a', 'n', 'i', 'a', 'ł', 'e', 'g', 'o', ' ', 'i', ' ', 'p', 'e', 'ł', 'n', 'e', 'g', 'o', ' ', 'e', 'n', 'e', 'r', 'g', 'i', 'i', ' ', 'd', 'n', 'i', 'a', '!', ' ', ':flexed_biceps:', ':smiling_face_with_smiling_eyes:']


In [60]:
# to może wcześniej podzielmy to na słowa
emoji_words = emoji_text.split()
emoji_corpus = [emoji.demojize(doc) for doc in emoji_words]
print("Tekst bez emoji:\n", emoji_corpus)

Tekst bez emoji:
 ['Kawa', 'zaparzona', ':hot_beverage:,', 'plany', 'na', 'dziś', 'zrobione', ':memo:.', 'Czas', 'ruszać', 'do', 'działania', 'i', 'łapać', 'chwile!', ':sparkles:', 'Życzę', 'Wam', 'wspaniałego', 'i', 'pełnego', 'energii', 'dnia!', ':flexed_biceps::smiling_face_with_smiling_eyes:']


#### Podsumowanie

Można dodać jeszcze wiele przekształceń takiego tekstu, takich jak:
* ekstrakcja adresów e-mail, numerów telefonów
* pozbycię się znaczników HTML
* redukcja wieloktornych znaków interpunkcyjnych, np. !!!, ???
* zamiana wielkości znaków
* podstawianie specjalnych znaczników w miejsce wybranych elementów, np. tagów, adresów stron, e-maili i wiele innych,
* usuwanie słów stop (stop words removal).

### Zadania - część 1

In [ ]:
# tekst do przetworzenia w ćwiczeniach

taka_historia = '<p> Cześć!!! 👋 Witajcie na moim nowym blogu o sztucznej inteligencji...   Dzisiaj porozmawiamy o NLP (Natural Language Processing). </p> 

Czy wiedzieliście, że aż 80% danych w firmach to dane nieustrukturyzowane??? 😲 Więcej informacji znajdziecie na stronie: https://www.przykladowastrona.pl/nlp-wstep lub pisząc na e-mail: kontakt@moj-blog-ai.com.pl. 

#MachineLearning #DataScience @Kowalski_Data_Geek 

W      niektórych    miejscach celowo zostawiłem  duuuuuużo spacji i tabulacji. ALBO NAPISAŁEM COŚ CAPSLOCKIEM, żebyście mieli co zmieniać na małe litery (tzw. lowercasing). 
Warto usunąć z tego tekstu polskie stop-words, np.: "i", "w", "na", "oraz", "że". 

Data publikacji: 12.05.2023 r., godz. 14:30. Zysk firmy wzrósł o $45,000 w Q3!
P.S. Nie zapomnijcie o usunięciu tagów HTML, np. <b>pogrubienia</b> i znaków interpunkcyjnych! 🚀'

**Zadanie 1**

* a) zamień wielkość znaków na znaki małe
* b) pozbądź się nadmiarowych białych znaków
* c) pozbądź się nadmiarowych znaków przestankowych następujących po sobie (!!!, ???). Możesz je znaleźć w module `string`.
* d) zamień emotikony na ich tekstową postać
* e) pozbądź się wszystkich tagów HTML

**Zadanie 2**

Masz do dyspozycji trzy tagi:
* `<NUM>` - liczby, daty, wartości księgowe i wszystko co stanowi wartość numeryczną,
* `<URL>` - adresy URL
* `<EMAIL>` - adresy email

Za pomocą wyrażeń regularnych i/lub funkcji wbudowanych klasy `str` zamień wartości w tekście na powyższe tagi.

**Zadanie 3**

Wykorzystując listę stop words z adresu https://github.com/bieli/stopwords/blob/master/polish.stopwords.txt pozbądź się wszystkich słów stop z tekstu.

## 2. Przykładowe zadania z biblioteką `spaCy`

In [ ]:
!pip install nltk spacy
!python -m spacy download pl_core_news_sm

In [62]:
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

# Załadowanie modelu polskiego
nlp = spacy.load("pl_core_news_sm")

### 2.1 Tokenizacja i stop words

Przepuszczenie surowego tekstu przez model `nlp` automatycznie dokonuje tokenizacji i tagowania części mowy.

In [63]:
text = "Studenci pilnie uczą się Pythona, ponieważ chcą zostać świetnymi programistami w 2026 roku!"
doc = nlp(text)

print("TOKENY i ich właściwości:")
for token in doc:
    print(f"{token.text:15} | Czy Stop-Word? {token.is_stop} | Czy Interpunkcja? {token.is_punct} | Czy Cyfra? {token.is_digit}")

TOKENY i ich właściwości:
Studenci        | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
pilnie          | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
uczą            | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
się             | Czy Stop-Word? True | Czy Interpunkcja? False | Czy Cyfra? False
Pythona         | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
,               | Czy Stop-Word? False | Czy Interpunkcja? True | Czy Cyfra? False
ponieważ        | Czy Stop-Word? True | Czy Interpunkcja? False | Czy Cyfra? False
chcą            | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
zostać          | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
świetnymi       | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
programistami   | Czy Stop-Word? False | Czy Interpunkcja? False | Czy Cyfra? False
w               | Czy Stop-Word? True | Czy Interpunk

#### ZADANIE 2.1:
Napisz funkcję `clean_text(raw_text)`, która przyjmie tekst, przetworzy go przez `nlp()`, a następnie zwróci string, z którego **usunięto** wszystkie tokeny będące:
1. znakami interpunkcyjnymi (`token.is_punct`),
2. liczbami (`token.is_digit`),
3. tzw. stop-wordami (`token.is_stop`).

*Wskazówka: Złóż ze sobą zachowane tokeny z powrotem w jeden string oddzielony spacją (`" ".join(...)`). Przetestuj na zmiennej `text`.*

### 2.2 Lematyzacja

Jak sprawić, by model traktował formy słów "chłopca", "chłopców", "chłopcem" jako jedną i tę samą koncepcję? Trzeba ujednolicić ich formę sprowadzając do mianownika (lematu).

In [64]:
text_lemmatization = "Rozumiałem te zadania, chociaż wczoraj kompletnie ich nie rozumiałam."
doc_lem = nlp(text_lemmatization)

print("ORYGINAŁ -> LEMAT")
for token in doc_lem:
    print(f"{token.text:15} -> {token.lemma_}")

ORYGINAŁ -> LEMAT
Rozumiałem      -> Rozumiałem
te              -> ten
zadania         -> zadanie
,               -> ,
chociaż         -> chociaż
wczoraj         -> wczoraj
kompletnie      -> kompletnie
ich             -> on
nie             -> nie
rozumiałam      -> rozumiać być
.               -> .


#### ZADANIE 2.2:
Rozbuduj swoją funkcję z Zadania 2.1 do nowej postaci `preprocess_text(raw_text)`. Niech funkcja nadal usuwa znaki interpunkcyjne, liczby oraz stopwords, **ALE** zamiast oryginalnego słowa (token.text), niech dodaje do ostatecznego wyniku jego ujednolicony lemat (`token.lemma_`). Przekształć w ten sposób wszystkie lematy na małe litery (użyj `.lower()`).

Przetestuj wynik na napisie: `"Szybkie psy goniły najszybsze koty po dachach budynków."`

### 2.3 Wektoryzacja: Od słów do macierzy liczb

Użyjemy klasycznego podejścia Scikit-Learn - narzędzia `CountVectorizer`.

In [65]:
corpus = [
    "Bardzo lubię programować w języku Python.",
    "Python jest super językiem.",
    "Nie znoszę, kiedy mój kod nie działa."
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

# Oglądamy macierz wyników BoW (Bag of Words)
df_bow = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
display(df_bow)

,bardzo,działa,jest,językiem,języku,kiedy,kod,lubię,mój,nie,programować,python,super,znoszę
0,1,0,0,0,1,0,0,1,0,0,1,1,0,0
1,0,0,1,1,0,0,0,0,0,0,0,1,1,0
2,0,1,0,0,0,1,1,0,1,2,0,0,0,1


Zwróć uwagę, że słowa odmienione (języku, językiem) traktowane są jako dwie odrębne cechy! Dlatego lematyzacja (Zadanie 2) jest zazwyczaj wymagana przed wpuszczeniem polskiego tekstu w wektoryzator.

#### ZADANIE 2.3:
Poniżej zadeklarowano mini-korpus recenzji. Twoim celem jest:
1. Nadpisanie każdej recenzji w liście (użyj pętli) za pomocą swojej funkcji z Zadania 2.2 (`preprocess_text`), by dokonać czyszczenia i lematyzacji.
2. Użycie `TfidfVectorizer` (zamiast CountVectorizer) na czystym korpusie.
3. Wyświetlenie nowej macierzy TF-IDF jako DataFrame Pandas (tak jak pokazano wyżej).

In [66]:
reviews = [
    "Telefon jest genialny, świetne zdjęcia robi.",
    "Nigdy więcej tego telefonu, zdjęcia są koszmarne!",
    "Telefon popsuł się po tygodniu, katastrofa i koszmar.",
    "Kocham te zdjęcia!"
]

**Zadanie 3 - dla chętnych**

Wykonaj oczyszczanie lektury "Calineczka", którą możesz pobrać uruchamiając kod z komórki poniżej. Zastanów się, które informacje z pliku są najistotniejsze w kontekście przygotowania korpusu do treningu modelu. Możesz wykorzystać również tokenizację i lematyzację analizując czy wszystkie słowa znalazły swoją formę podstawową. Opisz wnioski.

In [ ]:
!curl https://wolnelektury.pl/media/book/txt/calineczka.txt > calineczka.txt